# OOP Week 6 -- Domain Exceptions & Validation

**Course:** Object-Oriented Programming (Year 2)
**Session:** 3 hours
**Prerequisites:** Weeks 1-5
**Focus:** custom exception hierarchy, validation flow, error messages that help

---

## Learning Objectives

1. Explain why custom exceptions are better than generic ones
2. Build an exception hierarchy for a data pipeline
3. Add validation that raises informative exceptions
4. Use try/except to handle exceptions gracefully
5. Design exceptions that carry useful context

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Section 1: Why Custom Exceptions?

In your v2 code, errors probably looked like:
```python
raise ValueError("bad data")
```

This is unhelpful because:
- **Which** data was bad? Which row? Which column?
- **Why** was it bad? Missing? Out of range? Wrong type?
- **What should the user do** about it?

Custom exceptions solve all of these problems. They carry context-specific information and have descriptive names.

### Example: Generic vs Custom Exceptions

In [ ]:
# GENERIC (unhelpful)
try:
    raise ValueError("bad data")
except ValueError as e:
    print("Generic:", e)
    # What data? What was bad? What do I do?

print()

# CUSTOM (informative)
class SchemaError(Exception):
    def __init__(self, missing_columns, actual_columns):
        self.missing = missing_columns
        self.actual = actual_columns
        msg = ("Schema mismatch: missing " + str(missing_columns)
               + ". Available: " + str(actual_columns))
        super().__init__(msg)

try:
    raise SchemaError(["value", "timestamp"], ["id", "status"])
except SchemaError as e:
    print("Custom:", e)
    print("Missing:", e.missing)
    print("Actual: ", e.actual)

**Expected Output:**
```
Generic: bad data

Custom: Schema mismatch: missing ['value', 'timestamp']. Available: ['id', 'status']
Missing: ['value', 'timestamp']
Actual:  ['id', 'status']
```

---
## Section 2: Building an Exception Hierarchy

We create a **base** exception for our pipeline, then specific exceptions for each type of error. This lets us catch all pipeline errors at once or handle specific ones.

---
### Exception Hierarchy

```
Exception (built-in)
  |
  +-- PipelineError (our base)
        |
        +-- SchemaError (missing/wrong columns)
        |
        +-- CleaningError (too much data dropped)
        |
        +-- EmptyDataError (no data at a stage)
        |
        +-- ConfigError (bad configuration)
```

In [ ]:
class PipelineError(Exception):
    """Base exception for all pipeline errors."""
    pass


class SchemaError(PipelineError):
    """Data schema does not match expectations."""
    def __init__(self, missing_columns, actual_columns):
        self.missing = missing_columns
        self.actual = actual_columns
        msg = ("Schema mismatch: missing " + str(missing_columns)
               + ". Available: " + str(actual_columns))
        super().__init__(msg)


class CleaningError(PipelineError):
    """Too many rows dropped during cleaning."""
    def __init__(self, n_raw, n_clean, threshold=0.5):
        self.n_raw = n_raw
        self.n_clean = n_clean
        self.threshold = threshold
        if n_raw > 0:
            self.drop_rate = 1 - (n_clean / n_raw)
        else:
            self.drop_rate = 1.0
        pct = str(round(self.drop_rate * 100)) + "%"
        thr = str(round(threshold * 100)) + "%"
        msg = ("Cleaning dropped " + pct + " of data ("
               + str(n_raw) + " -> " + str(n_clean)
               + "). Threshold: " + thr)
        super().__init__(msg)


class EmptyDataError(PipelineError):
    """Pipeline received empty data."""
    def __init__(self, stage):
        self.stage = stage
        super().__init__("Empty data at stage: " + stage)


class ConfigError(PipelineError):
    """Invalid configuration."""
    def __init__(self, key, message):
        self.key = key
        super().__init__("Config error [" + key + "]: " + message)


print("Exception hierarchy defined!")
print("All inherit from PipelineError -> Exception")

**Expected Output:**
```
Exception hierarchy defined!
All inherit from PipelineError -> Exception
```

---
## Section 3: Using Custom Exceptions

In [ ]:
def validate_and_clean(data, config):
    """Full validation pipeline with custom exceptions."""

    # 1. Check for empty data
    if not data:
        raise EmptyDataError("load")

    # 2. Check schema
    required = config.get("required_columns", [])
    actual = list(data[0].keys())
    missing = set(required) - set(actual)
    if missing:
        raise SchemaError(list(missing), actual)

    # 3. Clean
    cleaned = [r for r in data if r.get("value") is not None]

    # 4. Check drop rate
    if len(data) > 0 and len(cleaned) < len(data) * 0.5:
        raise CleaningError(len(data), len(cleaned))

    return cleaned


# Test 1: Empty data
try:
    validate_and_clean([], {})
except EmptyDataError as e:
    print("Caught EmptyDataError:", e)

# Test 2: Schema mismatch
try:
    validate_and_clean([{"a": 1}], {"required_columns": ["value"]})
except SchemaError as e:
    print("Caught SchemaError:", e)

# Test 3: Too much dropped
try:
    data = [{"value": None}] * 8 + [{"value": 10}] * 2
    validate_and_clean(data, {})
except CleaningError as e:
    print("Caught CleaningError:", e)

# Test 4: Catch ALL pipeline errors
try:
    validate_and_clean([], {})
except PipelineError as e:
    print("Caught PipelineError (catches all subtypes):", type(e).__name__)

**Expected Output:**
```
Caught EmptyDataError: Empty data at stage: load
Caught SchemaError: Schema mismatch: missing ['value']. Available: ['a']
Caught CleaningError: Cleaning dropped 80% of data (10 -> 2). Threshold: 50%
Caught PipelineError (catches all subtypes): EmptyDataError
```

---
### Design Decision: When to raise vs when to handle

**Raise** an exception when:
- The error makes it impossible to continue
- The caller needs to know something went wrong
- You want to prevent bad data from propagating silently

**Handle** (try/except) when:
- You can recover from the error
- You want to log it and continue
- You are at the 'top level' and need to show the user a message

**Rule of thumb:** Low-level components RAISE. High-level orchestrators HANDLE.

---
### Common Mistake: Catching too broadly

The code below has a bug. Can you spot it before reading the fix?

In [ ]:
# BAD: catches EVERYTHING, including bugs!
try:
    result = 1 / 0  # this is a bug, not a data error
except Exception:
    print("Something went wrong")  # hides the real problem!

**What goes wrong:** Catching `Exception` hides bugs. Only catch specific exceptions that you know how to handle. Let unexpected errors crash -- they reveal bugs that need fixing.

**The fix:**

In [ ]:
# GOOD: catch only what you expect
try:
    validate_and_clean([], {})
except PipelineError as e:
    print("Data error:", e)  # only catches our errors
# ZeroDivisionError would still crash (good! it's a bug)

---
### Try It!

Add a `ConfigError` validation: before running the pipeline, check that the config has a `value_column` key. Raise `ConfigError` if missing.

In [ ]:
# YOUR CODE HERE


---
## Section 4: Exception-Aware Components

In [ ]:
class ValidatingDataSource:
    """DataSource that raises domain exceptions."""

    def __init__(self, path, required_columns=None):
        self.path = path
        self.required_columns = required_columns or []

    def load(self):
        # Simulate loading
        raw = [
            {"id": 1, "value": 25.0},
            {"id": 2, "value": 30.0},
        ]

        if not raw:
            raise EmptyDataError("load")

        if self.required_columns:
            actual = list(raw[0].keys())
            missing = set(self.required_columns) - set(actual)
            if missing:
                raise SchemaError(list(missing), actual)

        print("Loaded " + str(len(raw)) + " rows")
        return raw


# Happy path
vds = ValidatingDataSource("data.csv", required_columns=["id", "value"])
data = vds.load()
print("Success:", data)

# Error path
try:
    bad = ValidatingDataSource("data.csv", required_columns=["temperature"])
    bad.load()
except SchemaError as e:
    print("Caught: " + str(e))
    print("Missing columns:", e.missing)

**Expected Output:**
```
Loaded 2 rows
Success: [{'id': 1, 'value': 25.0}, {'id': 2, 'value': 30.0}]
Caught: Schema mismatch: missing ['temperature']. Available: ['id', 'value']
Missing columns: ['temperature']
```

---
## Section 5: Exception-Safe Pipeline

In [ ]:
class SafePipeline:
    """Pipeline that handles exceptions gracefully."""

    def __init__(self, source, cleaner):
        self.source = source
        self.cleaner = cleaner
        self.errors = []

    def run(self):
        """Run pipeline, collecting errors instead of crashing."""
        self.errors = []

        # Step 1: Load
        try:
            data = self.source.load()
        except PipelineError as e:
            self.errors.append(("load", e))
            print("LOAD FAILED: " + str(e))
            return None

        # Step 2: Clean
        try:
            clean = self.cleaner(data)
        except PipelineError as e:
            self.errors.append(("clean", e))
            print("CLEAN FAILED: " + str(e))
            return None

        print("Pipeline completed successfully")
        return clean

    def report_errors(self):
        if not self.errors:
            print("No errors")
            return
        for stage, error in self.errors:
            print("Error at " + stage + ": " + type(error).__name__
                  + " -- " + str(error))


# Test: successful run
source = ValidatingDataSource("data.csv", ["id", "value"])
pipeline = SafePipeline(source, lambda d: [r for r in d if r.get('value', 0) > 0])
result = pipeline.run()
print("Result:", result)
print()

# Test: error run
bad_source = ValidatingDataSource("data.csv", ["temperature"])
bad_pipeline = SafePipeline(bad_source, lambda d: d)
result = bad_pipeline.run()
bad_pipeline.report_errors()

**Expected Output:**
```
Loaded 2 rows
Pipeline completed successfully
Result: [{'id': 1, 'value': 25.0}, {'id': 2, 'value': 30.0}]

Loaded 2 rows
LOAD FAILED: Schema mismatch: missing ['temperature']. ...
Error at load: SchemaError -- ...
```

---
### Common Mistake: Swallowing exceptions silently

The code below has a bug. Can you spot it before reading the fix?

In [ ]:
# BAD: exception is caught but nothing is done about it!
try:
    data = validate_and_clean([], {})
except:
    pass  # silent failure -- data is undefined!

try:
    print("Data:", data)  # NameError!
except NameError as e:
    print("ERROR: data was never defined because exception was swallowed")

**What goes wrong:** Catching an exception with `except: pass` is the worst pattern. It hides bugs, makes debugging impossible, and can leave variables undefined. Always either handle the exception (log, fallback) or let it propagate.

**The fix:**

In [ ]:
# GOOD: handle the exception properly
try:
    data = validate_and_clean([], {})
except EmptyDataError as e:
    print("Cannot proceed: " + str(e))
    data = []  # provide a fallback

print("Data:", data)  # safe: data is always defined

---
### Debugging Tip: Reading exception tracebacks

When you see an error traceback, read it **bottom to top**:

1. The LAST line tells you what went wrong (the exception type and message)
2. The lines ABOVE show where it happened (file, line number, function)
3. Follow the chain upward to find the ROOT cause

Custom exceptions make step 1 much more informative:
- `ValueError: bad data` -- useless
- `SchemaError: missing ['temperature']. Available: ['id', 'value']` -- actionable!

---
### Try It!

Add a `ConfigError` to the hierarchy. It should store the config key that is problematic and a description. Use it in a `validate_config(config)` function.

In [ ]:
# YOUR CODE HERE


---
### Try It!

Create a `RetryablePipeline` that, when a load fails, waits and retries up to 3 times before giving up. Hint: use a for loop with try/except inside.

In [ ]:
# YOUR CODE HERE


---
## Build from Scratch Exercise

This exercise tests whether you truly understand this week's concepts. Complete it without looking at the examples above.

In [ ]:
# BUILD FROM SCRATCH:
# Build an exception hierarchy for a web API: ApiError (base), AuthenticationError, NotFoundError, ValidationError. Each carries relevant context (status code, endpoint, etc).

# YOUR CODE HERE


In [ ]:
# TEST your build-from-scratch code:

# YOUR TESTS HERE


---
## Connect the Dots

How does this week's concept connect to previous weeks?

In [ ]:
# Where in the pipeline (Week 5) should each exception type be raised?

# YOUR ANSWER (as comments or code):


---
## Real-World Spotting

OOP patterns are everywhere in real software. Can you spot them?

In [ ]:
# Visit any Python library's GitHub and find their custom exceptions. How are they organized? (e.g., requests library has ConnectionError, Timeout, etc.)

# YOUR ANSWER:


---
## Diagram It

Draw an ASCII class diagram for the main classes from this week. Include:
- Class names
- Key attributes
- Key methods
- Relationships (has-a, is-a)

In [ ]:
# Draw your ASCII diagram here:
# +------------------+
# |   ClassName      |
# +------------------+
# | - attribute      |
# +------------------+
# | + method()       |
# +------------------+

# YOUR DIAGRAM:


---
## Key Vocabulary

| Term | Definition |
|------|------------|
| **Exception hierarchy** | A tree of exception classes (base -> specific) |
| **Raise** | Create and throw an exception |
| **Catch** | Handle an exception with try/except |
| **Propagate** | Let an exception pass up to the caller |
| **Context** | Extra information attached to an exception |

---
## Recap Exercise

Without looking at the code above, try to:

In [ ]:
# 1. Write one class from this week FROM MEMORY
#    (it does not need to be perfect)

# YOUR CODE HERE


# 2. Create an instance and call at least one method

# YOUR CODE HERE


# 3. Write one test for your class

# YOUR CODE HERE


---
## What to Review Before Next Week

Before the next session, make sure you can:

1. Explain this week's main concept in your own words
2. Write a simple example from memory
3. Identify this pattern in existing code
4. Explain WHY this pattern is useful (not just HOW)

---
## Mini-Quiz

In [ ]:
# Q1: Why is raise ValueError('bad') worse than raise SchemaError(...)?
# Answer: 

# Q2: What does 'except PipelineError' catch?
# Answer: 

# Q3: When should you raise vs handle an exception?
# Answer: 

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)